In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import itertools

from seq_utils import generate_kv_mapping

# Constants

In [45]:
seq_folder = "/Users/ccnlab/Development/sequences/shaping/trans"

NUM_KEYS = 4
NUM_FOOD = 4
NUM_VILLAGERS = 6
NUM_TRANS = 4
ALL_VF_IMAGES_SET = 4
TRANS_FOLDER = 1
OUTPUT_COL_ORDER = [
    "stim",
    "correct_key",
    "block",
    "img_folder",
    "trans_folder",
    "key0_trans",
    "key1_trans",
    "key2_trans",
    "key3_trans",
    "shop0_food",
    "shop1_food",
    "shop2_food",
    "shop3_food",
    "trans0_shop",
    "trans1_shop",
    "trans2_shop",
    "trans3_shop",
    "set_size",
]

SELF_PACED_OUTPUT_COL_ORDER = [
    "stim",
    "correct_key",
    "img_folder",
    "set_size",
    "type",
    "trans_folder",
]

In [3]:
# First, create an array with numbers 0 to 5 in order (no repeats)
seq = []
for i in range(6):
    seq.extend(np.random.permutation(6))
seq

[4,
 5,
 0,
 1,
 2,
 3,
 2,
 0,
 1,
 3,
 5,
 4,
 4,
 0,
 1,
 5,
 3,
 2,
 1,
 3,
 5,
 2,
 0,
 4,
 1,
 4,
 2,
 5,
 0,
 3,
 0,
 1,
 4,
 2,
 3,
 5]

# Helpers

In [ ]:
from shaping_utils import generate_non_shaping_block, generate_shaping_block


def shuffle_with_consecutive_check(stim_seq, key_dir, idx_check=1):
    # Pair stim_seq and key_dir by index and shuffle the pairs
    # Shuffle pairs while ensuring no consecutive key_dir values
    max_attempts = 12000
    for attempt in range(max_attempts):
        paired_data = list(zip(stim_seq, key_dir))

        np.random.shuffle(paired_data)

        # Check if any consecutive elements have the same key_dir
        consecutive_same = False
        for i in range(len(paired_data) - 1):
            if (
                paired_data[i][idx_check] == paired_data[i + 1][idx_check]
            ):  # Compare key_dir values
                # Find a successive element with different key_dir to swap with
                swap_idx = None
                for j in range(i + 1, len(paired_data)):
                    if (paired_data[j][idx_check] != paired_data[i][idx_check]) and (
                        paired_data[j][idx_check] != paired_data[i - 1][idx_check]
                    ):
                        swap_idx = j
                        break

                if swap_idx is not None:
                    paired_data[i], paired_data[swap_idx] = (
                        paired_data[swap_idx],
                        paired_data[i],
                    )

            if (
                paired_data[i][idx_check] == paired_data[i + 1][idx_check]
            ):  # Compare key_dir values
                consecutive_same = True
                break

        if not consecutive_same:
            break
    else:
        print(
            f"Warning: Could not find arrangement without consecutive key_dir after {max_attempts} attempts"
        )
    return paired_data


def generate_top_transfer_block(
    trans_shop_mapping,
    last_num_stim_iter,
    num_villagers,
    previous_img_set,
    num_food=NUM_FOOD,
):
    stim_food_mapping = generate_kv_mapping(num_villagers, num_food)

    tranfer_block = generate_non_shaping_block(
        NUM_KEYS,
        num_food,
        last_num_stim_iter,
        trans_shop_mapping,
        stim_food_mapping,
    )
    choices = np.setdiff1d(np.arange(0, ALL_VF_IMAGES_SET), [previous_img_set])
    new_img_set = np.random.choice(choices)
    tranfer_block["img_folder"] = new_img_set + 1
    tranfer_block["trans_folder"] = TRANS_FOLDER
    tranfer_block["set_size"] = num_villagers
    tranfer_block["block"] = 1
    return tranfer_block[OUTPUT_COL_ORDER]


def generate_test_block(
    NUM_TEST_ITER, img_set, set_size, trans_shop_mapping, stim_food_mapping
):
    testing_data = generate_non_shaping_block(
        NUM_KEYS,
        NUM_FOOD,
        NUM_TEST_ITER,
        trans_shop_mapping,
        stim_food_mapping,
    )
    testing_data["block"] = 1
    testing_data["img_folder"] = img_set + 1
    testing_data["trans_folder"] = TRANS_FOLDER
    testing_data["set_size"] = set_size
    testing_data = testing_data[OUTPUT_COL_ORDER]
    return testing_data


def getnerate_self_paced_block(
    trans_shop_mapping, stim_food_mapping, set_size, img_set
):
    self_paced_seq_data = []
    for t, mapping in zip([0, 1], [trans_shop_mapping, stim_food_mapping]):
        seq_data = self_paced_seq(mapping)
        seq_data["type"] = t  # trans=0, food=1
        self_paced_seq_data.append(seq_data)

    self_paced_seq_data = pd.concat(self_paced_seq_data)
    self_paced_seq_data["set_size"] = set_size
    self_paced_seq_data["img_folder"] = img_set + 1
    self_paced_seq_data["trans_folder"] = TRANS_FOLDER
    return self_paced_seq_data[SELF_PACED_OUTPUT_COL_ORDER]


def self_paced_seq(sr_mapping):
    return pd.DataFrame({"stim": np.arange(len(sr_mapping)), "correct_key": sr_mapping})

# V2 Task sequences

In [ ]:
TOTAL_TRIALS_IN_SHAPING = 48

def generate_shaping_learning_round_v0518(
    last_nonshaping_block,
    trans_shop_mapping,
    stim_food_mapping,
    num_food=NUM_FOOD,
):
    num_trans = len(trans_shop_mapping)
    num_villagers = len(stim_food_mapping)
    num_trans_iter = int(TOTAL_TRIALS_IN_SHAPING / (2 * num_trans))
    shaping_block = generate_shaping_block(
        NUM_KEYS, num_food, num_trans_iter, trans_shop_mapping
    )
    shaping_block["block"] = 1

    num_iter_per_stim = int(TOTAL_TRIALS_IN_SHAPING / (2 * num_villagers))
    nonshaping_block = generate_non_shaping_block(
        NUM_KEYS,
        num_food,
        num_iter_per_stim,
        trans_shop_mapping,
        stim_food_mapping,
    )
    nonshaping_block["block"] = 2
    
    for col in ['img_folder', 'set_size', 'trans_folder']:
        val = last_nonshaping_block[col].iloc[0].astype(int)
        shaping_block[col] = val
        nonshaping_block[col] = val

    full_task_block = last_nonshaping_block.copy()
    full_task_block["block"] = 3
    return pd.concat([shaping_block, nonshaping_block, full_task_block])[OUTPUT_COL_ORDER]

In [46]:
# read previous mapping
VERSION = 0
data = pd.read_csv(f"{seq_folder}/shaping_trans{VERSION}_learning.csv")

# trans to shop mapping
trans_to_shop = np.full(NUM_TRANS, -1, dtype=int)
shop_to_trans = {}
for i in np.arange(4):
    trans_to_shop[i] = data[f"trans{i}_shop"].iloc[0].astype(int)
    shop_to_trans[data[f"trans{i}_shop"].iloc[0]] = i
print(trans_to_shop)
# stim to food mapping
nonshaped_block = data[data.block == 2]
stim_to_food = np.full(NUM_VILLAGERS, -1, dtype=int)
for stim in nonshaped_block.stim.unique():
    row = nonshaped_block[nonshaped_block.stim == stim].iloc[0]
    k = row.correct_key
    correct_shop = trans_to_shop[row[f"key{k}_trans"]]
    stim_to_food[row.stim] = row[f"shop{correct_shop}_food"]

stim_to_food

[1 0 3 2]


array([0, 1, 2, 3, 1, 3])

In [48]:
shaping_round = generate_shaping_learning_round_v0518(
    nonshaped_block, trans_to_shop, stim_to_food, NUM_FOOD
)

shaping_round.to_csv(
    f"{seq_folder}/shapingv0518_trans{VERSION}_learning.csv", index=False
)

# V1 Task sequences

In [5]:
def generate_shaping_round(
    bs,
    img_set,
    num_iter_per_stim,
    trans_shop_mapping,
    last_nonshaping_block,
    num_food=NUM_FOOD,
):
    shaping_block = generate_shaping_block(
        NUM_KEYS, num_food, num_iter_per_stim, trans_shop_mapping
    )

    shaping_block["block"] = bs * 2
    shaping_block["img_folder"] = img_set + 1
    shaping_block["set_size"] = last_nonshaping_block["set_size"]

    nonshaping_block = last_nonshaping_block.copy()
    nonshaping_block["block"] = bs * 2 + 1
    nonshaping_block["img_folder"] = img_set + 1
    return pd.concat([shaping_block, nonshaping_block])


def generate_nonshaping_round(
    bs,
    img_set,
    num_iter_per_stim,
    trans_shop_mapping,
    stim_food_mapping,
    last_nonshaping_block,
    num_food=NUM_FOOD,
):
    nonshaping_block = generate_non_shaping_block(
        NUM_KEYS,
        num_food,
        num_iter_per_stim,
        trans_shop_mapping,
        stim_food_mapping,
    )
    nonshaping_block["block"] = bs * 2
    nonshaping_block["img_folder"] = img_set + 1
    nonshaping_block["set_size"] = last_nonshaping_block["set_size"]

    last_block = last_nonshaping_block.copy()
    last_block["block"] = bs * 2 + 1
    last_block["img_folder"] = img_set + 1

    return pd.concat([nonshaping_block, last_block])


def generate_learning_round(
    last_num_stim_iter,
    num_villagers,
    img_set,
    iter_by_setsz={4: 12, 6: 8},
    num_trans=4,
    num_food=NUM_FOOD,
):
    shaping_blocks, nonshaping_blocks = [], []
    trans_shop_mapping = generate_kv_mapping(NUM_KEYS, NUM_KEYS)
    stim_food_mapping = generate_kv_mapping(num_villagers, num_food)

    last_nonshaping_block = generate_non_shaping_block(
        NUM_KEYS,
        num_food,
        last_num_stim_iter,
        trans_shop_mapping,
        stim_food_mapping,
    )
    last_nonshaping_block["set_size"] = num_villagers
    shaping_blocks = generate_shaping_round(
        0,
        img_set,
        iter_by_setsz[num_trans],
        trans_shop_mapping,
        last_nonshaping_block,
    )
    nonshaping_blocks = generate_nonshaping_round(
        0,
        img_set,
        iter_by_setsz[num_villagers],
        trans_shop_mapping,
        stim_food_mapping,
        last_nonshaping_block,
    )
    return (
        shaping_blocks,
        nonshaping_blocks,
        trans_shop_mapping,
        stim_food_mapping,
    )

In [ ]:
# testing round - two iteration per direction per stimulus
LAST_BLOCK_ITER = 18
SZ_TO_ITER = {4: 12, 6: 8}
NUM_TEST_ITER = 4

set_size = 6
for seq_idx in range(ALL_VF_IMAGES_SET):
    img_set = seq_idx  # np.random.choice(np.arange(0, ALL_VF_IMAGES_SET))
    shaping_blocks, nonshaping_blocks, trans_shop_mapping, stim_food_mapping = (
        generate_learning_round(
            LAST_BLOCK_ITER, set_size, img_set, SZ_TO_ITER, 4, NUM_FOOD
        )
    )
    tranfer_block = generate_top_transfer_block(
        trans_shop_mapping,
        LAST_BLOCK_ITER,
        set_size,
        img_set,
        NUM_FOOD,
    )
    testing_data = generate_test_block(
        NUM_TEST_ITER, img_set, set_size, trans_shop_mapping, stim_food_mapping
    )

    self_paced_seq_data = getnerate_self_paced_block(
        trans_shop_mapping, stim_food_mapping, set_size, img_set
    )

    for name, data in zip(
        ["shaping", "nonshaping"], [shaping_blocks, nonshaping_blocks]
    ):
        concated_data = data
        concated_data["block"] = concated_data["block"] + 1
        concated_data["trans_folder"] = TRANS_FOLDER
        concated_data = concated_data[OUTPUT_COL_ORDER]
    #     concated_data.to_csv(
    #         f"{seq_folder}/{name}_trans{seq_idx}_learning.csv", index=False
    #     )

    # tranfer_block.to_csv(f"{seq_folder}/trans{seq_idx}_transfer.csv", index=False)
    # testing_data.to_csv(f"{seq_folder}/trans{seq_idx}_testing.csv", index=False)
    # self_paced_seq_data.to_csv(
    #     f"{seq_folder}/trans{seq_idx}_self_paced_testing.csv", index=False
    # )